# 商品实体匹配：数据、特征与评价

[English](product-matching-walkthrough.ipynb) | **简体中文**

同一商品在两家零售商处，可能使用不同的标题。这个实验利用文本和字段比较来判断两条记录是否对应。先看已保存的实验结果，再用几个小例子理解输入处理和评价指标。

最终模型在本项目中简称 **C+**：RoBERTa 读取商品对文本，并融合42项字段比较，再平均三个训练种子的匹配概率。后面的对照表会分别说明各个方法。

真实结果来自 **555 对固定的 Walmart–Amazon 商品记录**。这是回顾性比较：历史接触记录不完整，Walmart–Amazon 独立新样本的表现尚未验证。下文的 Acme 商品、标签和概率均为教学编写，不是模型预测。

**运行方法**：选择 Python 3.11 及以上内核，按顺序运行单元。在仓库根目录运行 `python scripts/run_notebook.py`，可仅用标准库执行中英文两份讲解并核对保存输出。它不会下载数据或模型、安装依赖或训练模型。

[项目概览](../README.zh-CN.md) · [数据与结果](../showcase/zh-CN/index.html) · [复现指南](../docs/zh-CN/REPRODUCIBILITY.md)


In [1]:
from pathlib import Path
import json, math, sys
LANGUAGE = "zh-CN"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "results/fixed-results.json").is_file() and (p / "src/product_matching").is_dir())
sys.path.insert(0, str(ROOT / "src"))
facts = json.loads((ROOT / "results/fixed-results.json").read_text(encoding="utf-8"))
text = json.loads((ROOT / "notebooks/walkthrough_text.json").read_text(encoding="utf-8"))[LANGUAGE]
print(text["task"])
for role, count in facts["roles"].items():
    print(f'{text["roles"][role]}: {count:,}')
print(text["scope"])


任务：判断两家零售商的商品记录是否指向同一商品。
训练商品对: 3,738
开发商品对: 199
校准商品对: 209
评价商品对: 555
范围：这是回顾性结果；历史接触记录不完整，Walmart–Amazon 独立新样本的表现尚未验证。


## 1. 数据用途与标签分离

| 用途 | 商品对数量 | 做什么 |
| --- | ---: | --- |
| 训练 | 3,738 | 更新模型参数，拟合标题词语证据 |
| 开发 | 199 | 选择共同检查点轮数 |
| 校准 | 209 | 选择分类阈值 |
| 评价 | 555 | 衡量已冻结的模型与阈值 |

预测与标签通过 `pair_id` 对齐，不依赖行顺序。只有**训练用途的模型输入**携带目标标签；校准和评价标签由相应的指标步骤单独接收。

数据流是：`训练集拟合 IDF → 按用途准备输入 → 开发集选轮数 → 校准集选阈值 → 评价`。这些名称说明数据的预定用途，不能证明历史评价样本从未被接触。


## 2. 固定评价池的模型对照

下面读取的是公开保存的聚合结果，不是后面的合成例子。六种方法使用同一批 555 对样本。B22、L30、S33 和 L42 是对照方法；C+ 是最终神经模型集成，C0是独立训练的消融网络，训练和推断时均将42项特征置零。

先看 **AP**，它反映正例在排序中的位置；再看 **F1**，它在校准阈值下综合考虑误判与漏判；**P@100** 只看排序最前的 100 对。AP 不等于分类准确率。


In [2]:
print(f'{text["method"]:<8} {"AP":>9} {"F1":>9} {"P@100":>9}')
for method, result in facts["metrics"].items():
    print(f'{method:<8} {result["ap"]:9.6f} {result["f1"]:9.6f} {result["p_at_100"]:9.2f}')


方法              AP        F1     P@100
B22       0.795590  0.743719      0.81
L30       0.844899  0.762115      0.90
S33       0.846702  0.767123      0.89
L42       0.949700  0.852041      1.00
C+        0.965998  0.901639      1.00
C0        0.965616  0.875648      1.00


### 指标绝对差与百分点

L42 是这批样本中表现最好的表格模型对照。将 C+ 的分数减去 L42，再乘以 100，就得到以百分点表示的绝对差值。同时比较独立训练的C0，检查训练和推断时均不提供42项特征的网络表现。


In [3]:
cplus, reference, ablation = (facts["metrics"][m] for m in ("C+", "L42", "C0"))
print(f'{text["ap_gain"]}: {(cplus["ap"]-reference["ap"])*100:.4f} {text["pp"]}')
print(f'{text["f1_gain"]}: {(cplus["f1"]-reference["f1"])*100:.4f} {text["pp"]}')
print(f'{text["ap_c0"]}: {(cplus["ap"]-ablation["ap"])*100:.4f} {text["pp"]}')


相较 L42 的 AP 增量: 1.6298 个百分点
相较 L42 的 F1 增量: 4.9599 个百分点
相较 C0 的 AP 差值: 0.0381 个百分点


C+ 优于 L42，但它的 AP 与 C0 很接近。这组结果不能证明 42 项特征带来了明显的 AP 增益，也不能证明差异具有统计显著性。它描述的是这批固定样本，不是独立新来源上的泛化结果。


## 3. 合成商品对的特征构造

两条 Acme 记录是编写的例子，型号标点和标题空格有所不同。特征由 **14 项业务字段比较 + 8 项标题比较（4项训练数据拟合的IDF加权比较、4项直接比较）+ 8 项标题型号代码比较 + 12 项原生型号与类别比较**组成。输出只展示其中四项，不是商品是否匹配的预测。

例子中的两个价格都能解析，币种也都是 USD，因此可以比较。缺失价格保持缺失；币种未知或不同，则标为不可比较。紧凑字符串有助于比较 `ZX-100` 和 `ZX100`，但不能把看起来相似的 `ZX-101` 自动视为相同型号。

这里编写的训练标签只用于演示输入结构与 IDF 构建。**没有训练分类器，RoBERTa 也没有为这两条记录打分。**特征代码的位置见[代码导览](../docs/zh-CN/CODE_MAP.md)。


In [4]:
from product_matching.data import business_record, native_record
from product_matching.features import FEATURE_NAMES_42, fit_title_evidence, prepare_rows
raw_left = {"title":"Acme ZX-100 16GB", "brand":"Acme", "price":"19.99", "priceCurrency":"USD"}
raw_right = {"title":"ACME ZX100 16 GB", "brand":"ACME", "price":"20.00", "priceCurrency":"USD"}
example = {"pair_id":"synthetic:acme", "left":business_record(raw_left), "right":business_record(raw_right),
           "left_native":native_record({"modelno":"ZX-100", "category":"electronics"}),
           "right_native":native_record({"modelno":"ZX100", "category":"electronics"})}
fit_idf = fit_title_evidence([{**example, "label":1}])
prepared = prepare_rows([example], role="dev", fit_idf=fit_idf)
values = dict(zip(FEATURE_NAMES_42, prepared[0]["x42"]))
print(text["feature_width"], ":", len(values))
for key in ("brand_equal", "price_comparable", "title_compact_exact", "native_model_compact_exact"):
    print(f'{text["features"][key]} [{key}]: {values[key]:.6f}')
assert len(values)==42 and "label" not in prepared[0]


合成例子的特征数量 : 42
品牌一致 [brand_equal]: 1.000000
价格可比较 [price_comparable]: 1.000000
紧凑标题完全一致 [title_compact_exact]: 1.000000
紧凑型号完全一致 [native_model_compact_exact]: 1.000000


### 输入用途边界校验

尝试把目标标签放进评价输入。接口应拒绝它，并保持仅由训练数据拟合的 IDF 状态不变。这个检查验证的是当前接口规则，不能倒推出历史数据使用从未发生泄漏。


In [5]:
from product_matching.errors import AdmissionError
before = fit_idf.payload()
try:
    prepare_rows([{**example, "label":1}], role="evaluation", fit_idf=fit_idf)
except AdmissionError:
    print(text["label_rejected"])
else:
    raise AssertionError(text["label_accepted"])
assert before == fit_idf.payload()
print(text["idf_unchanged"])


评价输入按预期拒绝了目标标签。
仅由训练数据拟合的 IDF 状态没有改变。


## 4. AB/BA方向与三种子概率集成

RoBERTa-base 分别读取 AB 和 BA 两种商品顺序。64 维投影与 42 项比较特征拼接，再经过一个 64 维 GELU 层、dropout 和标量输出，得到 logit。每个种子先把 AB/BA 的 logit 取均值，再用 sigmoid 转成概率；三个种子的概率最后等权平均。

C+按六轮调度训练，开发数据选择共同第4轮检查点用于集成。集成保留全部三个种子。训练时编码器使用 BF16 自动混合精度，参数和分类头使用 FP32；推断使用 FP32。固定配置见[模型说明](../docs/zh-CN/MODEL_CARD.md)。

下一单元使用**编写的概率 0.9、0.3 和 0.3**讲解计算。它们不是 RoBERTa 输出，也不是真实种子的实验结果。


In [6]:
from product_matching.metrics import mean_scores
invented = {seed:[{"pair_id":"synthetic:acme", "score":value}]
            for seed,value in zip((42,43,44),(.9,.3,.3))}
mean = mean_scores(["synthetic:acme"], invented)[0]["score"]
print(text["mean"], ":", mean)
print(text["seeds"], ":", facts["model"]["seeds"], ";", text["epoch"], ":", facts["model"]["epoch"])
assert mean == .5


编写的示例概率：等权均值 : 0.5
实验随机种子 : [42, 43, 44] ; 共同训练轮数 : 4


## 5. 排序指标、阈值与混淆计数

AP 根据排序中精确率和召回率的变化计算。并列分数作为一组处理，不会因为其中某条记录排在前面就随意给它额外优势。下面编写的例子里，最高分并列的两条包含一个正例和一个负例，AP 为 0.583333；这不是分类准确率。

真实实验在校准数据上按 F1 选择阈值；并列时先看精确率，再选更高阈值。评价阶段不重新选择阈值。合成 AP 例子之后，代码读取 C+ **已保存的聚合混淆计数**，按 `2 × TP / (2 × TP + FP + FN)` 独立复算 F1。

TP 是找到的真实匹配，FP 是误判为匹配，FN 是漏掉的匹配，TN 是正确排除的不匹配。修改阈值可能改变这些计数和 F1，但不会改变原有排序的 AP。


In [7]:
from product_matching.metrics import average_precision
toy = [{"pair_id":"synthetic:a", "label":1, "score":.9},
       {"pair_id":"synthetic:b", "label":0, "score":.9},
       {"pair_id":"synthetic:c", "label":1, "score":.5}]
print(f'{text["toy_ap"]}: {average_precision(toy):.6f}')
print(text["threshold"], ":", cplus["threshold"])
confusion = cplus["confusion"]
tp,fp,fn,tn = (confusion[k] for k in ("tp","fp","fn","tn"))
f1_from_counts = 2*tp/(2*tp+fp+fn)
assert math.isclose(f1_from_counts, cplus["f1"], rel_tol=0, abs_tol=1e-14)
print(f'{text["confusion"]}: TP={tp}, FP={fp}, FN={fn}, TN={tn}')
print(f'{text["f1_recovered"]}: {f1_from_counts:.6f}')


编写的示例分数：并列情况下的 AP: 0.583333
已保存的 C+ 校准阈值 : 0.3947772259513537
已保存的聚合计数: TP=165, FP=11, FN=25, TN=354
由已保存计数复算的 F1: 0.901639


## 6. 复现范围与执行命令

| 检查 | 已有证据覆盖什么 |
| --- | --- |
| 公开轻量检查与这份 Notebook | 聚合结果的算术、输入规则和合成例子；无需权重或商品数据集 |
| 现有权重回放 | 在 Windows RTX 3060 笔记本上，20 条命令产生的 23 组结果与原输出一致，预测差值为零 |
| 原始 CSV 重建与完整重训 | 尚未完成从头到尾的验收 |

回放使用本地提供的原始产物，并重建模型输入。它确认新代码入口可以重放已有模型，没有训练新模型，也没有使用新评价标签。公开仓库不分发商品记录或训练好的权重。

运行 `python scripts/public_smoke.py` 和 `python scripts/run_notebook.py` 可进行轻量检查。实际运行模型的单独命令见[复现指南](../docs/zh-CN/REPRODUCIBILITY.md)，解释边界见[局限说明](../docs/zh-CN/LIMITATIONS.md)。这份 Notebook 不会调用这些 GPU 命令。


In [8]:
for key, value in facts["reproduction"].items():
    display = text["yes"] if value is True else text["no"] if value is False else value
    print(f'{text["evidence"][key]}: {display}')
heavy = {"torch", "transformers", "sklearn", "sentence_transformers"}
assert not heavy & set(sys.modules)
print(text["done"])


回放记录中的核心测试数: 48
现有权重回放的命令数: 20
独立核对的结果组数: 23
最大预测差值: 0.0
回放期间是否重新训练: 否
是否已验收完整重训: 否
执行完成：只使用公开聚合结果和合成例子，没有加载模型库、读取商品数据集、推断或训练。


## 理解检查

1. **AP 0.9660 是否表示分类准确率为 96.60%，P@100 = 1 是否表示全部商品对都判对了？**都不是。AP 衡量排序，P@100 只覆盖排序最前的 100 对。
2. **Acme 例子是否运行了 RoBERTa？现有权重回放是否重新训练了模型？**都没有。Acme 演示输入处理；回放加载已经训练好的参数。
3. **能否用评价标签选择一个表现更好的阈值？**不能。应在校准数据上选择，并在评价前冻结。

接着可以阅读[项目卡](../docs/zh-CN/PROJECT_CARD.md)或[模型与证据的局限](../docs/zh-CN/LIMITATIONS.md)。
